# Tuning part of MPW3 - Image Captioning (IC)

To shorten the report-notebook we outsourced the tuning part to this notebook.

##  Team Name and Members
* Team Name: **demidova_keller** (used in submitted file names)
* Student 1: Iuliia Demidova (iuliia.demidova@students.fhnw.ch)
* Student 2: Lucas Keller (lucas.keller@students.fhnw.ch)

In [ ]:
# imports
import sys
from pathlib import Path
# fixing imports for provided_sources/ modules
sys.path.append(str(Path().resolve().parents[1])) 
# I had to add this section above for importing the provided_sources modules (Lucas)

# stdlib
import copy
import json
import math
import random
import textwrap
import time
from collections import defaultdict
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any

# third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import wandb
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from PIL import Image
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

# local project imports
import ic_configs
from dataloader import build_tokenizer_from_split, create_caption_dataloader
from tokenizer import tokenize
from notebooks.ic.ic_utils import (
    _filter_wandb_metrics,
    _is_better,
    _wandb_safe,
    build_references_by_image_id,
    collect_fixed_eval_subset,
    decode_token_ids,
    evaluate_bleu,
    evaluate_loss,
    generate_batch_token_ids,
    seed_everything,
    train_one_epoch,
)

print("torch version:", torch.__version__)
print("torchvision version:", torchvision.__version__)

torch version: 2.10.0+cu128
torchvision version: 0.25.0+cu128


In [3]:
# Reproducibility / seed
SEED = 13
seed_everything(SEED)

# Device choice
DEVICE = None
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("device:", DEVICE)

device: cuda


## W&B sweep for hyperparameter tuning model 1:

Since the model is located in the notebook `captioning_demidova_keller.ipynb` and not in a .py file, we simply copied the necessary parts directly into this notebook. This is not a good solution and carries the risk of code discrepancies between the two notebooks, as the code should be identical. In any case, the hyperparameter optimization was performed in this manner.

The following sections cover the necessary code snippets:

In [4]:
# data dir in the same dir as NB
data_dir = "data/"
# Tokenizer / dataloader parameters
# Given dataset is relatively small, so we:
#   filter out rare words to reduce vocabulary noise,
#   keep captions long enough to avoid cutting normal sentences,
#   and use a batch size that is okay for CPU/MPS

MIN_WORD_FREQ = 5   # Keep words appearing at least 5 times in train captions
MAX_LEN = 40        # Max caption length including <bos>/<eos>, longer captions are cut
BATCH_SIZE = 32     # Kind of default value

In [5]:
# Transforms
IMAGENET_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def _convert_to_rgb(image: Image.Image) -> Image.Image:
    return image.convert("RGB")

def build_train_transforms(augmentation= "medium") -> transforms.Compose:
    """choos the strength of the augmentation:
    - none: only resize and normalize.
    - light: resize, random crop, horizointal flip.
    - medium: random rotation, random resizecrop, random horizontal flip, color jitter.
    """
    if augmentation == "none":
        return transforms.Compose([
            transforms.Lambda(_convert_to_rgb),
            transforms.Resize((IMAGENET_SIZE, IMAGENET_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    if augmentation == "light":
        return transforms.Compose([
            transforms.Lambda(_convert_to_rgb),
            transforms.Resize((256, 256)),
            transforms.RandomCrop((IMAGENET_SIZE, IMAGENET_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    if augmentation == "medium":
        return transforms.Compose([
            transforms.Lambda(_convert_to_rgb),
            transforms.RandomApply([
                transforms.RandomRotation(degrees=3),
            ], p=0.25),
            transforms.RandomResizedCrop(
                size=IMAGENET_SIZE,
                scale=(0.85, 1.0),
                ratio=(0.9, 1.1),
            ),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(
                brightness=0.15,
                contrast=0.15,
                saturation=0.10,
                hue=0.02,
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    raise ValueError(f"Unknown augmentation: {augmentation}")

def build_test_transforms() -> transforms.Compose:
    return transforms.Compose([
            transforms.Lambda(_convert_to_rgb),
            transforms.Resize((IMAGENET_SIZE,IMAGENET_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

def denormalize_image(image: torch.Tensor) -> torch.Tensor:
    """Undo ImageNet normalization for visualization."""
    mean_tensor = image.new_tensor(IMAGENET_MEAN).view(-1, 1, 1)
    std_tensor = image.new_tensor(IMAGENET_STD).view(-1, 1, 1)
    return (image * std_tensor + mean_tensor).clamp(0, 1)

In [6]:
class ShowAndTellEncoder(nn.Module):
    """ResNet18 image encoder producing one global image feature vector."""

    def __init__(
        self,
        encoder_name: str = "resnet18",
        pretrained: bool = True,
        freeze_encoder: bool = True,
    ):
        super().__init__()

        if encoder_name != "resnet18":
            raise ValueError(f"Only resnet18 is supported: {encoder_name}")

        weights = ResNet18_Weights.DEFAULT if pretrained else None
        backbone = resnet18(weights=weights)

        self.output_dim = backbone.fc.in_features  # 512 for ResNet18
        self.freeze_encoder = freeze_encoder

        # remove avg classifier head
        # keep conv body + global avg pooling
        self.cnn = nn.Sequential(*list(backbone.children())[:-1])

        if self.freeze_encoder:
            for param in self.cnn.parameters():
                param.requires_grad = False
            self.cnn.eval()

    def train(self, mode: bool = True):
        """Keep frozen encoder in eval mode so BatchNorm stats do not drift"""
        super().train(mode)
        if self.freeze_encoder:
            self.cnn.eval()
        return self

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        if self.freeze_encoder:
            with torch.no_grad():
                features = self.cnn(images)
        else:
            features = self.cnn(images)

        return features.flatten(start_dim=1)  # [B, 512]


class ShowAndTellCaptioner(nn.Module):
    """
    Show and Tell baseline.

    Training:
        image + decoder_inputs -> next-token logits

    Generation:
        image + <bos> -> token -> token -> ... -> <eos>
    """

    def __init__(self, model_cfg):
        super().__init__()

        required = {
            "vocab_size": model_cfg.vocab_size,
            "pad_idx": model_cfg.pad_idx,
            "bos_idx": model_cfg.bos_idx,
            "eos_idx": model_cfg.eos_idx,
        }
        missing = [name for name, value in required.items() if value is None]
        if missing:
            raise ValueError(f"Model config is missing tokenizer created fields: {missing}")

        self.vocab_size = model_cfg.vocab_size
        self.pad_idx = model_cfg.pad_idx
        self.bos_idx = model_cfg.bos_idx
        self.eos_idx = model_cfg.eos_idx

        self.encoder = ShowAndTellEncoder(
            encoder_name=model_cfg.encoder_name,
            pretrained=model_cfg.pretrained,
            freeze_encoder=model_cfg.freeze_encoder,
        )

        self.image_projection = nn.Linear(self.encoder.output_dim, model_cfg.embed_dim)

        self.embedding = nn.Embedding(
            num_embeddings=model_cfg.vocab_size,
            embedding_dim=model_cfg.embed_dim,
            padding_idx=model_cfg.pad_idx,
        )

        lstm_dropout = model_cfg.dropout if model_cfg.num_lstm_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=model_cfg.embed_dim,
            hidden_size=model_cfg.hidden_dim,
            num_layers=model_cfg.num_lstm_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )

        self.dropout = nn.Dropout(model_cfg.dropout)
        self.output_layer = nn.Linear(model_cfg.hidden_dim, model_cfg.vocab_size)

    def forward(self, images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: [B, 3, H, W]
            captions: [B, T] decoder inputs, usually captions[:, :-1]

        Returns:
            logits: [B, T, vocab_size]
        """
        image_features = self.encoder(images)                         # [B, 512]
        image_embedding = self.image_projection(image_features)        # [B, E]
        image_embedding = image_embedding.unsqueeze(1)                 # [B, 1, E]

        word_embeddings = self.embedding(captions)                     # [B, T, E]

        # paper-style conditioning: image embedding is the first LSTM input
        lstm_inputs = torch.cat([image_embedding, word_embeddings], dim=1)  # [B, T+1, E]

        lstm_outputs, _ = self.lstm(lstm_inputs)                       # [B, T+1, H]

        # ignore output after image input; use outputs after caption-prefix tokens
        token_outputs = lstm_outputs[:, 1:, :]                         # [B, T, H]

        logits = self.output_layer(self.dropout(token_outputs))         # [B, T, vocab_size]
        return logits

    @torch.no_grad()
    def generate(
        self,
        images: torch.Tensor,
        max_len: int = 35,
        bos_idx: int | None = None,
        eos_idx: int | None = None,
    ) -> torch.Tensor:
        """
        Greedy decoding.

        Returns:
            generated token ids without <bos>, but possibly including <eos>.
            Shape: [B, <= max_len]
        """
        self.eval()

        bos_idx = self.bos_idx if bos_idx is None else bos_idx
        eos_idx = self.eos_idx if eos_idx is None else eos_idx

        batch_size = images.size(0)
        device = images.device

        image_features = self.encoder(images)
        image_embedding = self.image_projection(image_features).unsqueeze(1)

        # first LSTM step consumes image embedding
        _, hidden = self.lstm(image_embedding)

        current_tokens = torch.full(
            (batch_size,),
            fill_value=bos_idx,
            dtype=torch.long,
            device=device,
        )

        generated = []
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
        eos_fill = torch.full_like(current_tokens, fill_value=eos_idx)

        for _ in range(max_len):
            word_embedding = self.embedding(current_tokens).unsqueeze(1)
            output, hidden = self.lstm(word_embedding, hidden)

            logits = self.output_layer(output.squeeze(1))
            next_tokens = logits.argmax(dim=-1)

            # add eos after sequence is finished
            next_tokens = torch.where(finished, eos_fill, next_tokens)

            generated.append(next_tokens)
            finished |= next_tokens.eq(eos_idx)

            if finished.all():
                break

            current_tokens = next_tokens

        if not generated:
            return torch.empty(batch_size, 0, dtype=torch.long, device=device)

        return torch.stack(generated, dim=1)

# NOT USED
# model_1 = ShowAndTellCaptioner(cfg_1.model).to(DEVICE)

# optimizer_1 = cfg_1.optimizer.cls(
#     (p for p in model_1.parameters() if p.requires_grad),
#     **cfg_1.optimizer.kwargs,
# )

# scheduler_1 = (
#     None
#     if cfg_1.scheduler.cls is None
#     else cfg_1.scheduler.cls(optimizer_1, **cfg_1.scheduler.kwargs)
# )

# total_params_1 = sum(p.numel() for p in model_1.parameters())
# trainable_params_1 = sum(p.numel() for p in model_1.parameters() if p.requires_grad)

# print(model_1)
# print(f"Total params:     {total_params_1:,}")
# print(f"Trainable params: {trainable_params_1:,}")


To find suitable hyperparameters we used the W&B sweep functionality. The following Hyperparameters are to be tuned:
* Learning rate (LR)
* Weight decay (WD)
* Dropout (DO)

The tunung will take place in three steps, first tuning the LR and WD and then with them fixed tune the DA and then DO.\
For the sweep we usewd the `grid`-search method, since its functionality is easier to interpret and reproducible. Eventough the `bayes` method would wioth higher probability lead to the "better" hyperparameters while being more efficient (https://en.wikipedia.org/wiki/Bayesian_optimization).

<b>Tuning LR and WD:</b>


For this part we used the DA-setting "medium", since we want to use data augmentation later on.


1. Sweep:

|Hyperparameters|Values                 |
|--------------:|----------------------:|
|LR             | 3e-3, 1e-3, 3e-4, 1e-4|
|WD             |  1e-3, 1e-4, 1e-5, 0.0|

2. Sweep:

|Hyperparameters|Values            |
|--------------:|-----------------:|
|LR             |1.5e-3, 1e-3, 7e-4|
|WD             |   1e-5, 1e-6, 0.0|


In [7]:
sweep_config = {
    "method": "grid", # alternatives: "random", "bayes"
    "metric": {
        "name": "val/loss",
        "goal": "minimize",
    },
    "parameters": {
        "lr": {
            "values": [7e-4, 1e-3, 1.5e-3]
        },
        "weight_decay": {
            "values": [0, 1e-6, 1e-5]
        },
    },
}

In [12]:
# RUN ONLY ONCE TO CREATE SWEEP!!!
sweep_id = wandb.sweep(
    sweep=sweep_config,
    project="MPW-IC",
    entity="MSE_DeLearn_SPR26", 
)

print(sweep_id)

Create sweep with ID: s5j2ok5h
Sweep URL: https://wandb.ai/MSE_DeLearn_SPR26/MPW-IC/sweeps/s5j2ok5h
s5j2ok5h


In [9]:
sweep_id = "MSE_DeLearn_SPR26/MPW-IC/emigi0vz"

In [13]:
def sweep_train():
    bs = 64

    with wandb.init(reinit=True) as wandb_run:

        sweep_cfg = wandb.config

        seed_everything(13)

        # fresh config per run
        cfg_1 = ic_configs.make_show_and_tell_config()
        cfg_1.train.device = "cuda"
        cfg_1.train.epochs = 20  # shorter for sweep
        cfg_1.train.early_stopping = False
        cfg_1.train.best_metric = "val/bleu4"
        cfg_1.train.best_mode = "max"
        cfg_1.evaluation.compute_bleu_every_n_epochs = 2
        cfg_1.evaluation.max_bleu_batches = 5

        cfg_1.dataset.dataset_dir = Path(data_dir)

        # inject sweep values
        cfg_1.optimizer.kwargs["lr"] = sweep_cfg.lr
        cfg_1.optimizer.kwargs["weight_decay"] = sweep_cfg.weight_decay

        # transforms
        cfg_1.dataset.train_transform = build_train_transforms()
        cfg_1.dataset.eval_transform = build_test_transforms()

        # tokenizer
        tokenizer = build_tokenizer_from_split(
            split=cfg_1.caption_data.train_split,
            data_dir=cfg_1.dataset.dataset_dir,
            min_freq=cfg_1.tokenizer.min_word_freq,
        )

        ic_configs.attach_tokenizer_to_model_config(cfg_1.model, tokenizer)

        # loaders
        train_loader = create_caption_dataloader(
            split=cfg_1.caption_data.train_split,
            data_dir=cfg_1.dataset.dataset_dir,
            tokenizer=tokenizer,
            transform=cfg_1.dataset.train_transform,
            batch_size=bs,
            max_len=cfg_1.tokenizer.max_len,
            caption_sampling=cfg_1.caption_data.train_caption_sampling,
            num_workers=0,
            pin_memory=True,
        )

        test_loader = create_caption_dataloader(
            split=cfg_1.caption_data.test_split,
            data_dir=cfg_1.dataset.dataset_dir,
            tokenizer=tokenizer,
            transform=cfg_1.dataset.eval_transform,
            batch_size=bs,
            max_len=cfg_1.tokenizer.max_len,
            caption_sampling=cfg_1.caption_data.eval_caption_sampling,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
        )

        test_image_loader = create_caption_dataloader(
            split=cfg_1.caption_data.test_split,
            data_dir=cfg_1.dataset.dataset_dir,
            tokenizer=tokenizer,
            transform=cfg_1.dataset.eval_transform,
            batch_size=bs,
            max_len=cfg_1.tokenizer.max_len,
            caption_sampling=cfg_1.caption_data.generation_caption_sampling,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
        )

        references_by_image_id = build_references_by_image_id(test_loader)

        # fresh model per sweep run
        model_1 = ShowAndTellCaptioner(cfg_1.model).to(DEVICE)

        optimizer_1 = cfg_1.optimizer.cls(
            model_1.parameters(),
            **cfg_1.optimizer.kwargs,
        )

        for epoch in range(1, cfg_1.train.epochs + 1):

            train_metrics = train_one_epoch(
                model=model_1,
                loader=train_loader,
                optimizer=optimizer_1,
                cfg=cfg_1,
                desc=f"sweep train {epoch}/{cfg_1.train.epochs}",
            )

            val_metrics = evaluate_loss(
                model=model_1,
                loader=test_loader,
                cfg=cfg_1,
                desc=f"sweep eval {epoch}/{cfg_1.train.epochs}",
            )

            metrics = {
                "epoch": epoch,
                "train/loss": train_metrics["loss"],
                "train/perplexity": train_metrics["perplexity"],
                "val/loss": val_metrics["loss"],
                "val/perplexity": val_metrics["perplexity"],
                "lr": optimizer_1.param_groups[0]["lr"],
                "weight_decay": cfg_1.optimizer.kwargs["weight_decay"],
            }

            # BLEU-scores
            if epoch % cfg_1.evaluation.compute_bleu_every_n_epochs == 0:

                bleu_metrics = evaluate_bleu(
                    model=model_1,
                    image_loader=test_image_loader,
                    references_by_image_id=references_by_image_id,
                    tokenizer=tokenizer,
                    cfg=cfg_1,
                    max_batches=cfg_1.evaluation.max_bleu_batches,
                )

                metrics.update({
                    "val/bleu1": bleu_metrics["bleu_1"],
                    "val/bleu2": bleu_metrics["bleu_2"],
                    "val/bleu3": bleu_metrics["bleu_3"],
                    "val/bleu4": bleu_metrics["bleu_4"],
                })

            wandb.log(metrics)

            print(
                f"epoch={epoch:03d} | "
                f"lr={cfg_1.optimizer.kwargs['lr']} | "
                f"wd={cfg_1.optimizer.kwargs['weight_decay']} | "
                f"train_loss={metrics['train/loss']:.4f} | "
                f"val_loss={metrics['val/loss']:.4f}"
            )

In [ ]:
wandb.agent(
    sweep_id=sweep_id,
    function=sweep_train,
    # count=1, # test one run for debugging
)

wandb: Agent Starting Run: ph8pqnrd with config:
wandb: 	lr: 0.0007
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lucas\_netrc.
wandb: Currently logged in as: lucas-j-keller98 (MSE_DeLearn_SPR26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


build references:   0%|          | 0/127 [00:00<?, ?it/s]

sweep train 1/20:   0%|          | 0/506 [00:00<?, ?it/s]

<b>Sweep Results:</b>
|Hyperparameter| Value|
|-------------:|-----:|
|Best LR       |`7e-4`|
|Best WD       | `0.0`|

(full insight can be seen in W&B sweeps g0ak4kpw & emigi0vz)

<b>Tune the LSTM dropout:</b>

Here we make a short sweep through three DO vlues:
* `0.3`
* `0.5`
* `0.6`

Higher values wouldn't make sense since th Network conist of a large frozen part that does/cannot change and there only is one LSTM-Layer. IN this particular case too much regularization would hurt performance.

In [ ]:
sweep_config = {
    "method": "grid",
    "metric": {
        "name": "val/loss",
        "goal": "minimize",
    },
    "parameters": {
        "dropout": {
            "values": [0.3, 0.5, 0.6]
        }
    },
}

In [ ]:
# RUN ONLY ONCE TO CREATE SWEEP!!!
sweep_id = wandb.sweep(
    sweep=sweep_config,
    project="MPW-IC",
    entity="MSE_DeLearn_SPR26", 
)

print(sweep_id)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lucas\_netrc.


Create sweep with ID: scqzb8ly
Sweep URL: https://wandb.ai/MSE_DeLearn_SPR26/MPW-IC/sweeps/scqzb8ly
scqzb8ly


In [ ]:
sweep_id = "MSE_DeLearn_SPR26/MPW-IC/scqzb8ly"

In [ ]:
def sweep_train():
    bs = 64

    with wandb.init(reinit=True) as wandb_run:

        sweep_cfg = wandb.config

        seed_everything(13)

        # fresh config per run
        cfg_1 = ic_configs.make_show_and_tell_config()
        cfg_1.train.device = "cuda"
        cfg_1.train.epochs = 20  # shorter for sweep
        cfg_1.train.early_stopping = False
        cfg_1.train.best_metric = "val/loss"
        cfg_1.train.best_mode = "min"
        cfg_1.evaluation.compute_bleu_every_n_epochs = 2
        cfg_1.evaluation.max_bleu_batches = 10

        cfg_1.dataset.dataset_dir = Path(data_dir)

        # initiate dropout from sweep config
        cfg_1.model.dropout = sweep_cfg.dropout

        # LR and WD obtained from prior sweeps
        cfg_1.optimizer.kwargs["lr"] = 7e-4
        cfg_1.optimizer.kwargs["weight_decay"] = 0.0

        # transforms
        cfg_1.dataset.train_transform = build_train_transforms("medium") # from prior sweeps
        cfg_1.dataset.eval_transform = build_test_transforms()

        # tokenizer
        tokenizer = build_tokenizer_from_split(
            split=cfg_1.caption_data.train_split,
            data_dir=cfg_1.dataset.dataset_dir,
            min_freq=cfg_1.tokenizer.min_word_freq,
        )

        ic_configs.attach_tokenizer_to_model_config(cfg_1.model, tokenizer)

        # loaders
        train_loader = create_caption_dataloader(
            split=cfg_1.caption_data.train_split,
            data_dir=cfg_1.dataset.dataset_dir,
            tokenizer=tokenizer,
            transform=cfg_1.dataset.train_transform,
            batch_size=bs,
            max_len=cfg_1.tokenizer.max_len,
            caption_sampling=cfg_1.caption_data.train_caption_sampling,
            num_workers=0,
            pin_memory=True,
        )

        test_loader = create_caption_dataloader(
            split=cfg_1.caption_data.test_split,
            data_dir=cfg_1.dataset.dataset_dir,
            tokenizer=tokenizer,
            transform=cfg_1.dataset.eval_transform,
            batch_size=bs,
            max_len=cfg_1.tokenizer.max_len,
            caption_sampling=cfg_1.caption_data.eval_caption_sampling,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
        )

        test_image_loader = create_caption_dataloader(
            split=cfg_1.caption_data.test_split,
            data_dir=cfg_1.dataset.dataset_dir,
            tokenizer=tokenizer,
            transform=cfg_1.dataset.eval_transform,
            batch_size=bs,
            max_len=cfg_1.tokenizer.max_len,
            caption_sampling=cfg_1.caption_data.generation_caption_sampling,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
        )

        references_by_image_id = build_references_by_image_id(test_loader)

        # fresh model per sweep run
        model_1 = ShowAndTellCaptioner(cfg_1.model).to(DEVICE)

        optimizer_1 = cfg_1.optimizer.cls(
            (p for p in model_1.parameters() if p.requires_grad),
            **cfg_1.optimizer.kwargs,
        )

        for epoch in range(1, cfg_1.train.epochs + 1):

            train_metrics = train_one_epoch(
                model=model_1,
                loader=train_loader,
                optimizer=optimizer_1,
                cfg=cfg_1,
                desc=f"sweep train {epoch}/{cfg_1.train.epochs}",
            )

            val_metrics = evaluate_loss(
                model=model_1,
                loader=test_loader,
                cfg=cfg_1,
                desc=f"sweep eval {epoch}/{cfg_1.train.epochs}",
            )

            metrics = {
                "epoch": epoch,
                "train/loss": train_metrics["loss"],
                "train/perplexity": train_metrics["perplexity"],
                "val/loss": val_metrics["loss"],
                "val/perplexity": val_metrics["perplexity"],
                "lr": optimizer_1.param_groups[0]["lr"],
                "weight_decay": cfg_1.optimizer.kwargs["weight_decay"],
            }

            # BLEU-scores
            if epoch % cfg_1.evaluation.compute_bleu_every_n_epochs == 0:

                bleu_metrics = evaluate_bleu(
                    model=model_1,
                    image_loader=test_image_loader,
                    references_by_image_id=references_by_image_id,
                    tokenizer=tokenizer,
                    cfg=cfg_1,
                    max_batches=cfg_1.evaluation.max_bleu_batches,
                )

                metrics.update({
                    "val/bleu1": bleu_metrics["bleu_1"],
                    "val/bleu2": bleu_metrics["bleu_2"],
                    "val/bleu3": bleu_metrics["bleu_3"],
                    "val/bleu4": bleu_metrics["bleu_4"],
                })

            wandb.log(metrics)

            print(
                f"epoch={epoch:03d} | "
                f"lr={cfg_1.optimizer.kwargs['lr']} | "
                f"wd={cfg_1.optimizer.kwargs['weight_decay']} | "
                f"train_loss={metrics['train/loss']:.4f} | "
                f"val_loss={metrics['val/loss']:.4f}"
            )

In [ ]:
wandb.agent(
    sweep_id=sweep_id,
    function=sweep_train,
    # count=1, # test one run for debugging
)

wandb: Agent Starting Run: gnqcjrln with config:
wandb: 	dropout: 0.3
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lucas\_netrc.
wandb: Currently logged in as: lucas-j-keller98 (MSE_DeLearn_SPR26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


build references:   0%|          | 0/127 [00:00<?, ?it/s]

sweep train 1/20:   0%|          | 0/506 [00:00<?, ?it/s]

sweep eval 1/20:   0%|          | 0/127 [00:00<?, ?it/s]

epoch=001 | lr=0.0007 | wd=0.0 | train_loss=3.9480 | val_loss=3.3842


...

sweep eval 20/20:   0%|          | 0/127 [00:00<?, ?it/s]

BLEU generation:   0%|          | 0/26 [00:00<?, ?it/s]

epoch=020 | lr=0.0007 | wd=0.0 | train_loss=2.0994 | val_loss=2.7980


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
train/perplexity,█▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val/bleu1,▁▄▄▇█▇█▇█▇
val/bleu2,▁▄▄▇█▇█▇█▇
val/bleu3,▁▄▄▇█▇█▇█▇
val/bleu4,▁▄▄▇█▇█▇█▇
val/loss,█▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▂▁▁▂
val/perplexity,█▅▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+1,...


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


<b>Sweep Results:</b>

The best results were achieved with the drouout rate of `0.6`. (fulll insight in W&B sweep scqzb8ly, somehow the last run got deleted - but believe me it was by a small margin better than `0.5`)

#### Final results of hyperparameter tuning:

To summarize the following table shows all the tuned hyparparameters and it's values:

|Hyperparameter|   Value|
|-------------:|-------:|
| LR           |  `7e-4`|
| WD           |   `0.0`|
| DO           |   `0.6`|

With these values a regular model (see: `captioning_demidova_keller.ipynb`) is fully traind to observe its training behaviour and early stopping is then applied on the final model to prevent overfitting. Which was already observed in the sweeps. Additionaly we are using a Lr-scheduler that reduces the LR by a factor of `0.5` if the threshold of `1e-3` is not exceede for `4` epochs (with a fixed min lr-rate of `1e-6`). Even though we saw that after 20 epoch the model ends up in an overfitting regime we set the number of training epochs to 30, because the BLEU-metrics tends to stabelise later than the validation-loss and -perplexity. 